# Quantum Drift Forecasting — Unified Pipeline and Comparative Analysis

This notebook presents the **complete, end-to-end Quantum Drift Forecasting pipeline** in a single reproducible artifact. It trains all four model architectures (VanillaRNN, LSTM, GRU, Transformer) on the full multi-qubit hardware telemetry dataset, conducts a comprehensive comparative evaluation, and demonstrates the practical workflow for **drift-aware recalibration scheduling**.

---

## Contents

1. [Setup and Configuration](#1.-Setup-and-Configuration)
2. [Full Multi-Qubit Dataset](#2.-Full-Multi-Qubit-Dataset)
3. [Training All Models](#3.-Training-All-Models)
4. [Comparative Forecasting Performance](#4.-Comparative-Forecasting-Performance)
5. [Comparative Drift Classification](#5.-Comparative-Drift-Classification)
6. [Anomaly Detection across All Qubits](#6.-Anomaly-Detection-across-All-Qubits)
7. [Recalibration Trigger Simulation](#7.-Recalibration-Trigger-Simulation)
8. [Statistical Analysis of Results](#8.-Statistical-Analysis-of-Results)
9. [Pipeline Summary and Reproducibility](#9.-Pipeline-Summary-and-Reproducibility)

---

> **Scientific context.** This project is directly relevant to U.S. national interests in quantum computing infrastructure. The practical success of quantum computing depends not only on theoretical breakthroughs, but also on the reliable operation of real quantum devices over time. Statistical learning contributes here by enabling forecasting, uncertainty quantification, anomaly detection, sequential inference, and drift-aware prediction — transforming quantum hardware from a fragile experimental platform into a more measurable, predictable, and operable technology.

## 1. Setup and Configuration

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

from src.data import (
    load_or_generate, build_dataset, FEATURE_COLS
)
from src.models import (
    VanillaRNN, LSTMForecaster, GRUForecaster, TransformerForecaster, AnomalyDetector
)
from src.evaluate import (
    forecast_metrics, classification_metrics,
    plot_forecast, plot_anomaly_scores,
    run_mc_dropout, conformal_margin, plot_model_comparison
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}  |  PyTorch {torch.__version__}')

SEED    = 42
SEQ_LEN = 32
HORIZON = 8
EPOCHS  = 40
BATCH   = 64
ALPHA   = 0.7   # forecast vs. classification loss weight

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs('../outputs', exist_ok=True)

## 2. Full Multi-Qubit Dataset

The combined dataset aggregates sequences from all 5 qubits into a single training pool. This increases the effective sample count and allows the models to learn cross-qubit generalisation patterns.

In [ ]:
df = load_or_generate('../data/quantum_device_metrics.csv')
print(f'Raw dataset: {df.shape[0]} rows × {df.shape[1]} cols')
print(f'Qubits: {df["qubit_id"].nunique()}  |  Time steps per qubit: {df["qubit_id"].value_counts().iloc[0]}')
print(f'Overall drift fraction: {df["drift_label"].mean():.3f}')

ds = build_dataset(
    csv_path='../data/quantum_device_metrics.csv',
    seq_len=SEQ_LEN,
    horizon=HORIZON,
)

Xtr, ytr, ltr = ds['train']
Xv,  yv,  lv  = ds['val']
Xte, yte, lte = ds['test']

INPUT_DIM = Xtr.shape[-1]
print(f'\nCombined training set  : {Xtr.shape}')
print(f'Combined validation set: {Xv.shape}')
print(f'Combined test set      : {Xte.shape}')
print(f'Input feature dimension: {INPUT_DIM}')
print(f'Drift fraction (train) : {ltr.mean():.3f}')
print(f'Drift fraction (test)  : {lte.mean():.3f}')

def make_loader(X, yf, yl, shuffle=False):
    return DataLoader(
        TensorDataset(
            torch.tensor(X,  dtype=torch.float32, device=device),
            torch.tensor(yf, dtype=torch.float32, device=device),
            torch.tensor(yl, dtype=torch.float32, device=device),
        ), batch_size=BATCH, shuffle=shuffle
    )

train_loader = make_loader(Xtr, ytr, ltr, shuffle=True)
val_loader   = make_loader(Xv,  yv,  lv)
test_loader  = make_loader(Xte, yte, lte)

In [ ]:
# Feature distribution overview
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
colors = ['#6366f1', '#34d399', '#f59e0b', '#f87171', '#818cf8', '#fb923c', '#a78bfa']

for i, col in enumerate(FEATURE_COLS):
    axes[i].hist(df[col], bins=40, color=colors[i], alpha=0.8, edgecolor='none')
    axes[i].set_title(col, color='#c7d2fe', fontsize=9)
    axes[i].set_xlabel('Value', fontsize=8)

axes[-1].axis('off')
plt.suptitle('Hardware Metric Distributions (all qubits, all times)', color='#e2e8f0', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Training All Models

In [ ]:
def train_model(model, name, train_loader, val_loader, epochs=EPOCHS, lr=1e-3):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_mae, best_state = float('inf'), None
    history = {'train_loss': [], 'val_mae': []}

    for epoch in tqdm(range(1, epochs + 1), desc=f'{name}', leave=True):
        model.train()
        ep = 0.0
        for Xb, yf_b, yl_b in train_loader:
            opt.zero_grad()
            fc, lg = model(Xb)
            loss = ALPHA * nn.functional.mse_loss(fc, yf_b) + \
                   (1 - ALPHA) * nn.functional.binary_cross_entropy_with_logits(lg.squeeze(-1), yl_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep += loss.item() * len(Xb)
        ep /= len(train_loader.dataset)
        sched.step()

        model.eval()
        v_mae = 0.0
        with torch.no_grad():
            for Xb, yf_b, _ in val_loader:
                fc, _ = model(Xb)
                v_mae += (fc - yf_b).abs().mean().item() * len(Xb)
        v_mae /= len(val_loader.dataset)

        if v_mae < best_mae:
            best_mae = v_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        history['train_loss'].append(ep)
        history['val_mae'].append(v_mae)

    model.load_state_dict(best_state)
    print(f'  {name}: best val MAE = {best_mae:.5f}')
    return history

models = {
    'RNN':         VanillaRNN(INPUT_DIM,     hidden_dim=64,  horizon=HORIZON, dropout=0.1).to(device),
    'LSTM':        LSTMForecaster(INPUT_DIM, hidden_dim=128, num_layers=2, horizon=HORIZON, dropout=0.2).to(device),
    'GRU':         GRUForecaster(INPUT_DIM,  hidden_dim=128, num_layers=2, horizon=HORIZON, dropout=0.2).to(device),
    'Transformer': TransformerForecaster(INPUT_DIM, d_model=128, nhead=4, num_layers=3,
                                         dim_ff=256, horizon=HORIZON, dropout=0.1).to(device),
}

for name, m in models.items():
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:12s}: {n:>8,} parameters')

In [ ]:
histories = {}
lrs = {'RNN': 1e-3, 'LSTM': 1e-3, 'GRU': 1e-3, 'Transformer': 5e-4}
for name, model in models.items():
    histories[name] = train_model(model, name, train_loader, val_loader,
                                  epochs=EPOCHS, lr=lrs[name])

In [ ]:
# Learning curves overview
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors_m = {'RNN': '#f59e0b', 'LSTM': '#6366f1', 'GRU': '#34d399', 'Transformer': '#f87171'}

for name, hist in histories.items():
    axes[0].plot(hist['train_loss'], label=name, color=colors_m[name], linewidth=1.3)
    axes[1].plot(hist['val_mae'],    label=name, color=colors_m[name], linewidth=1.3)

axes[0].set_title('Training Loss (all models)', color='#c7d2fe', fontsize=11)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()
axes[1].set_title('Validation MAE (all models)', color='#c7d2fe', fontsize=11)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE'); axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/combined_learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Comparative Forecasting Performance

In [ ]:
def get_all_preds(model, loader):
    model.eval()
    fc_l, lg_l, yf_l, yl_l = [], [], [], []
    with torch.no_grad():
        for Xb, yf_b, yl_b in loader:
            fc, lg = model(Xb)
            fc_l.append(fc.cpu().numpy());
            lg_l.append(lg.cpu().numpy())
            yf_l.append(yf_b.cpu().numpy());
            yl_l.append(yl_b.cpu().numpy())
    return (
        np.concatenate(fc_l),
        np.concatenate(lg_l).squeeze(-1),
        np.concatenate(yf_l),
        np.concatenate(yl_l),
    )

all_metrics = {}
all_preds   = {}
for name, model in models.items():
    fc, logits, y_true_f, y_true_l = get_all_preds(model, test_loader)
    all_preds[name] = (fc, logits, y_true_f, y_true_l)
    fm = forecast_metrics(y_true_f, fc)
    cm = classification_metrics(y_true_l, logits)
    all_metrics[name] = {**fm, **cm}

results_df = pd.DataFrame(all_metrics).T
print('Comparative test-set performance:')
results_df.round(5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric, title in [
    (axes[0], 'MAE',       'Forecast MAE (lower is better)'),
    (axes[1], 'RMSE',      'Forecast RMSE'),
    (axes[2], 'ROC-AUC',   'Drift Detection ROC-AUC (higher is better)'),
]:
    names  = list(all_metrics.keys())
    values = [all_metrics[n][metric] for n in names]
    bars   = ax.bar(names, values, color=[colors_m[n] for n in names], width=0.55)
    ax.bar_label(bars, fmt='%.4f', padding=3, color='#e2e8f0', fontsize=8)
    ax.set_title(title, color='#c7d2fe', fontsize=10, pad=8)
    ax.set_ylabel(metric)

plt.tight_layout()
plt.savefig('../outputs/combined_bar_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Horizon-wise MAE for all models
fig, ax = plt.subplots(figsize=(10, 4))

for name, model in models.items():
    fc, _, y_true_f, _ = all_preds[name]
    step_maes = [np.abs(y_true_f[:, h] - fc[:, h]).mean() for h in range(HORIZON)]
    ax.plot(range(1, HORIZON + 1), step_maes, marker='o', markersize=5,
            label=name, color=colors_m[name], linewidth=1.8)

ax.set_title('MAE vs Forecast Horizon — All Models (test set)', color='#c7d2fe', fontsize=12)
ax.set_xlabel('Forecast horizon (steps ahead)')
ax.set_ylabel('MAE (normalised T1)')
ax.set_xticks(range(1, HORIZON + 1))
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/combined_horizon_mae.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Comparative Drift Classification

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curves
for name in models:
    _, logits, _, y_true_l = all_preds[name]
    prob = 1 / (1 + np.exp(-logits))
    fpr, tpr, _ = roc_curve(y_true_l, prob)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})', color=colors_m[name], linewidth=1.8)

axes[0].plot([0, 1], [0, 1], '--', color='#475569', linewidth=1)
axes[0].set_title('ROC Curves — Drift Classification', color='#c7d2fe', fontsize=11)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend(fontsize=9)

# Precision-Recall curves
from sklearn.metrics import precision_recall_curve, average_precision_score
for name in models:
    _, logits, _, y_true_l = all_preds[name]
    prob = 1 / (1 + np.exp(-logits))
    prec, rec, _ = precision_recall_curve(y_true_l, prob)
    ap = average_precision_score(y_true_l, prob)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=colors_m[name], linewidth=1.8)

axes[1].set_title('Precision-Recall Curves — Drift Classification', color='#c7d2fe', fontsize=11)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/combined_roc_pr.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Anomaly Detection across All Qubits

We train a separate `AnomalyDetector` on the combined dataset and evaluate per-qubit anomaly scores, demonstrating how reconstruction-based detection generalises across the qubit register.

In [ ]:
# Train anomaly detector on normal (stable) sequences from all qubits
ad = AnomalyDetector(input_dim=INPUT_DIM, d_model=64, nhead=4, num_layers=2, dim_ff=128).to(device)
normal_Xtr = Xtr[ltr == 0]

ad_loader  = DataLoader(
    TensorDataset(torch.tensor(normal_Xtr, dtype=torch.float32, device=device)),
    batch_size=32, shuffle=True
)
ad_opt   = torch.optim.AdamW(ad.parameters(), lr=1e-3, weight_decay=1e-4)
ad_sched = torch.optim.lr_scheduler.CosineAnnealingLR(ad_opt, T_max=30)

for epoch in tqdm(range(30), desc='Training AnomalyDetector'):
    ad.train()
    for (Xb,) in ad_loader:
        ad_opt.zero_grad()
        loss = nn.functional.mse_loss(ad(Xb), Xb)
        loss.backward()
        nn.utils.clip_grad_norm_(ad.parameters(), 1.0)
        ad_opt.step()
    ad_sched.step()

ad.eval()
Xte_t = torch.tensor(Xte, dtype=torch.float32, device=device)
scores = ad.anomaly_scores(Xte_t).cpu().numpy()

# Threshold: 90th percentile of training scores
Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=device)
threshold = float(np.percentile(ad.anomaly_scores(Xtr_t).cpu().numpy(), 90))

print(f'Train-set threshold (P90): {threshold:.6f}')
print(f'Test normal score (mean)  : {scores[lte==0].mean():.6f}')
print(f'Test drift score (mean)   : {scores[lte==1].mean():.6f}')

fig = plot_anomaly_scores(
    scores[:120], lte[:120], threshold=threshold,
    title='Reconstruction-Based Anomaly Scores (Combined Dataset, first 120 windows)'
)
plt.savefig('../outputs/combined_anomaly_scores.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Recalibration Trigger Simulation

We simulate a **threshold-based recalibration policy** driven by the LSTM drift probability:  when the model's predicted drift probability exceeds a configurable threshold `p_thresh`, a recalibration event is triggered. We measure:

- **True recalibration rate**: fraction of actual drift events correctly intercepted.
- **False alarm rate**: fraction of triggered recalibrations that were unnecessary.
- **Calibration efficiency**: total recalibration count vs. a fixed-interval schedule.

In [ ]:
_, logits_lstm, _, y_true_l = all_preds['LSTM']
drift_prob_lstm = 1 / (1 + np.exp(-logits_lstm))

# Sweep threshold values
thresholds  = np.linspace(0.1, 0.9, 17)
true_recal  = []   # TPR: actual drift windows that trigger recalibration
false_alarm = []   # FPR: stable windows that trigger recalibration
total_triggers = []

n_drift  = (y_true_l == 1).sum()
n_stable = (y_true_l == 0).sum()

for p in thresholds:
    triggered = (drift_prob_lstm >= p).astype(int)
    tp = ((triggered == 1) & (y_true_l == 1)).sum()
    fp = ((triggered == 1) & (y_true_l == 0)).sum()
    true_recal.append(tp / n_drift   if n_drift   > 0 else 0)
    false_alarm.append(fp / n_stable if n_stable > 0 else 0)
    total_triggers.append(triggered.sum())

# Fixed-interval baseline: triggers every N steps
FIXED_INTERVAL = 5
fixed_triggers = len(y_true_l) // FIXED_INTERVAL
fixed_tp = 0
for i in range(0, len(y_true_l), FIXED_INTERVAL):
    window = y_true_l[i : i + FIXED_INTERVAL]
    if window.any():
        fixed_tp += 1
fixed_tpr = fixed_tp / (len(y_true_l) // FIXED_INTERVAL + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(thresholds, true_recal,  marker='o', markersize=5, color='#34d399', linewidth=1.8,
             label='True recal. rate (TPR)')
axes[0].plot(thresholds, false_alarm, marker='s', markersize=5, color='#f87171', linewidth=1.8,
             label='False alarm rate (FPR)')
axes[0].axhline(fixed_tpr, color='#f59e0b', linestyle='--', linewidth=1.2, label='Fixed-interval TPR')
axes[0].set_title('Recalibration Policy: TPR vs FPR (LSTM)', color='#c7d2fe', fontsize=11)
axes[0].set_xlabel('Drift probability threshold'); axes[0].legend(fontsize=9)

axes[1].plot(thresholds, total_triggers, marker='D', markersize=5, color='#818cf8', linewidth=1.8)
axes[1].axhline(fixed_triggers, color='#f59e0b', linestyle='--', linewidth=1.2,
                label=f'Fixed-interval ({fixed_triggers} triggers)')
axes[1].set_title('Total Recalibration Triggers vs Threshold', color='#c7d2fe', fontsize=11)
axes[1].set_xlabel('Drift probability threshold'); axes[1].set_ylabel('Trigger count')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/recalibration_policy.png', dpi=120, bbox_inches='tight')
plt.show()

# Optimal threshold (maximise TPR - FPR)
scores_thresh = np.array(true_recal) - np.array(false_alarm)
opt_idx   = int(np.argmax(scores_thresh))
opt_thresh = float(thresholds[opt_idx])
print(f'Optimal threshold: {opt_thresh:.2f}  →  TPR={true_recal[opt_idx]:.3f}, FPR={false_alarm[opt_idx]:.3f}')
print(f'Fixed-interval baseline TPR: {fixed_tpr:.3f}')
print(f'Model triggers at opt. threshold: {total_triggers[opt_idx]}  vs  fixed: {fixed_triggers}')

## 8. Statistical Analysis of Results

In [ ]:
from scipy import stats

# Bootstrap confidence intervals for test MAE of LSTM vs Transformer
np.random.seed(SEED)
N_BOOT = 1000
N_TE   = len(Xte)

def bootstrap_mae(y_true, y_pred, n=N_BOOT):
    maes = []
    for _ in range(n):
        idx = np.random.randint(0, len(y_true), len(y_true))
        maes.append(np.abs(y_true[idx, 0] - y_pred[idx, 0]).mean())
    return np.array(maes)

boot_lstm = bootstrap_mae(all_preds['LSTM'][2], all_preds['LSTM'][0])
boot_tf   = bootstrap_mae(all_preds['Transformer'][2], all_preds['Transformer'][0])

print(f'LSTM        MAE: {boot_lstm.mean():.5f}  95% CI [{np.percentile(boot_lstm, 2.5):.5f}, {np.percentile(boot_lstm, 97.5):.5f}]')
print(f'Transformer MAE: {boot_tf.mean():.5f}   95% CI [{np.percentile(boot_tf, 2.5):.5f}, {np.percentile(boot_tf, 97.5):.5f}]')

t_stat, p_val = stats.ttest_rel(boot_lstm, boot_tf)
print(f'\nPaired t-test LSTM vs Transformer MAE: t={t_stat:.3f}, p={p_val:.4f}')
if p_val < 0.05:
    winner = 'LSTM' if boot_lstm.mean() < boot_tf.mean() else 'Transformer'
    print(f'Result: statistically significant difference (α=0.05). {winner} achieves lower MAE.')
else:
    print('Result: no statistically significant difference at α=0.05.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([boot_lstm, boot_tf],
           labels=['LSTM', 'Transformer'],
           patch_artist=True,
           boxprops=dict(facecolor='#1e2130', color='#6366f1'),
           medianprops=dict(color='#34d399', linewidth=2),
           whiskerprops=dict(color='#94a3b8'),
           capprops=dict(color='#94a3b8'),
           flierprops=dict(marker='o', color='#6366f1', alpha=0.4))
ax.set_title('Bootstrap MAE Distribution (1000 resamples)', color='#c7d2fe', fontsize=11)
ax.set_ylabel('1-step forecast MAE')
plt.tight_layout()
plt.savefig('../outputs/bootstrap_mae_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Pipeline Summary and Reproducibility

In [ ]:
# Save comprehensive results table
results_df = pd.DataFrame(all_metrics).T.round(5)
results_df.to_csv('../outputs/combined_model_results.csv')
print('Saved: outputs/combined_model_results.csv')
print('\nFinal comparative results:')
print(results_df.to_string())

In [ ]:
# Visualise key outputs
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)

# --- Top row: forecast comparison ---
for col, name in enumerate(['RNN', 'LSTM', 'Transformer']):
    ax = fig.add_subplot(gs[0, col])
    fc, _, y_true_f, _ = all_preds[name]
    ax.plot(y_true_f[:40, 0], color='#6366f1', linewidth=1.2, label='Truth', alpha=0.9)
    ax.plot(fc[:40, 0],       color='#34d399', linewidth=1.2, linestyle='--', label='Forecast')
    ax.set_title(f'{name} — 1-step Forecast', color='#c7d2fe', fontsize=9, pad=5)
    ax.set_xlabel('Test step', fontsize=8)
    if col == 0: ax.set_ylabel('Normalised T1', fontsize=8)
    ax.legend(fontsize=7)

# --- Bottom row: horizon MAE, ROC, anomaly scatter ---
ax4 = fig.add_subplot(gs[1, 0])
for name in models:
    fc, _, y_true_f, _ = all_preds[name]
    step_maes = [np.abs(y_true_f[:, h] - fc[:, h]).mean() for h in range(HORIZON)]
    ax4.plot(range(1, HORIZON+1), step_maes, marker='o', markersize=4,
             label=name, color=colors_m[name], linewidth=1.3)
ax4.set_title('MAE vs Horizon', color='#c7d2fe', fontsize=9, pad=5)
ax4.set_xlabel('Horizon step', fontsize=8); ax4.set_ylabel('MAE', fontsize=8)
ax4.legend(fontsize=7)

ax5 = fig.add_subplot(gs[1, 1])
for name in models:
    _, logits, _, y_true_l = all_preds[name]
    prob = 1 / (1 + np.exp(-logits))
    fpr, tpr, _ = roc_curve(y_true_l, prob)
    roc_auc = auc(fpr, tpr)
    ax5.plot(fpr, tpr, label=f'{name} ({roc_auc:.3f})', color=colors_m[name], linewidth=1.3)
ax5.plot([0,1],[0,1],'--',color='#475569',linewidth=1)
ax5.set_title('ROC Curves', color='#c7d2fe', fontsize=9, pad=5)
ax5.set_xlabel('FPR', fontsize=8); ax5.set_ylabel('TPR', fontsize=8)
ax5.legend(fontsize=7)

ax6 = fig.add_subplot(gs[1, 2])
normal_scores = scores[lte == 0]
drift_scores  = scores[lte == 1]
ax6.hist(normal_scores, bins=25, alpha=0.7, color='#34d399', label='Stable', density=True)
ax6.hist(drift_scores,  bins=25, alpha=0.7, color='#f87171', label='Drifting', density=True)
ax6.axvline(threshold, color='#f59e0b', linestyle='--', linewidth=1.5, label='Threshold')
ax6.set_title('Anomaly Score Distribution', color='#c7d2fe', fontsize=9, pad=5)
ax6.set_xlabel('Score', fontsize=8); ax6.legend(fontsize=7)

fig.suptitle('Quantum Drift Forecasting — Combined Pipeline Summary', color='#e2e8f0', fontsize=12, y=1.01)
plt.savefig('../outputs/combined_pipeline_summary.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: outputs/combined_pipeline_summary.png')

In [ ]:
# Environment and reproducibility report
import platform, datetime

print('═' * 60)
print('  QUANTUM DRIFT FORECASTING — PIPELINE SUMMARY')
print('═' * 60)
print(f'  Date      : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'  Python    : {platform.python_version()}')
print(f'  PyTorch   : {torch.__version__}')
print(f'  Device    : {device}')
print(f'  Seed      : {SEED}')
print(f'  Seq len   : {SEQ_LEN}')
print(f'  Horizon   : {HORIZON}')
print(f'  Features  : {INPUT_DIM} ({FEATURE_COLS})')
print(f'  Train N   : {len(Xtr)}')
print(f'  Test N    : {len(Xte)}')
print()
print('  MODEL RESULTS (test set):')
for name, m in all_metrics.items():
    print(f'  {name:12s}  MAE={m["MAE"]:.4f}  RMSE={m["RMSE"]:.4f}  F1={m["F1"]:.4f}  AUC={m["ROC-AUC"]:.4f}')
print()
print('  OUTPUT FILES:')
for f in sorted(os.listdir('../outputs')):
    print(f'    outputs/{f}')
print('═' * 60)